In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "eckert2017great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Eckert_2014_Final_Results_Summary_tab_1.csv")
complete_path_2 = os.path.join(original_data_pathway, "Eckert_2014_tab_preference_single_item_Final_Results_Summary.csv")
complete_path_3 = os.path.join(original_data_pathway, "Eckert_2014_tab_preference_population_Final_Results_Summary.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df3 = pd.read_csv(complete_path_3)

# df1['experiment_name']="Final_Results"

df2['preference_type']="preference_single_item"
df2['experiment']=0

df3['preference_type']="preference_population"
df3['experiment']=0


In [3]:

data_frames=[ df1, df2, df3]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s)
    x = x.rename(columns={"subject": "ape",
        "age [years]": "age",
        "order of conditions": "order_of_conditions"})
    x['study_id']="eckert2017great"
    data_frames[index]=x
new_df=data_frames[0]


In [4]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)
# fulldf.columns


In [5]:
fulldf=fulldf.rename(columns={"group": "group_original"})

fulldf[['group','group1']] = fulldf['group_original'].str.split('-',expand=True)

group_replace = ['orangs', 'gorillas', 'bonobos']
for x in group_replace:
    fulldf['group'] = fulldf['group'].str.replace(x, '')



In [6]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')


In [7]:
fulldf = fulldf.rename(columns={"species_y": "species",
    "sex_y": "sex",
    "preference_single-item ": "preference_single_item",
    "preference_pop-open": "preference_pop_open",
    "preference_pop-covered": "preference_pop_covered",
    "position pellet pop.": "position_pellet_pop",
    "choice [pellet/carrot]": "choice_pellet_or_carrot"})

fulldf.rename(columns={"ape": "participant", "group":"species_subgroup",
                       "age":"age_original"}, inplace=True)


In [8]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

# fulldf.columns

In [9]:
fulldf=fulldf[['study_id','experiment',  'preference_type',  'day', 'month', 'year',
         'participant', 'age_original','age_in_years',  'sex', 'species', 'species_subgroup', 'session', 'trial', 'condition','order_of_conditions', 
       'preference_single_item',
       'preference_pop_open', 'preference_pop_covered', 
       'initial_position_pellet_pop',
       'position_pellet_sample', 'choice', 
       'preference', 'correct choice', 'position_pellet_pop',
       'choice_pellet_or_carrot' ]]


In [10]:
fulldf['experiment']= fulldf['experiment'].astype(str)
fulldf['experiment'].unique()

array(['1', '2', '0'], dtype=object)

In [11]:
exp1 = fulldf[fulldf['experiment'] == '1'] 
exp2 = fulldf[fulldf['experiment'] == '2']
exp3 = fulldf[fulldf['experiment'] == '0']

experiments = [[exp1, 'eckert2017great_exp1'], ##connects df with name of output dataset
                [ exp2, 'eckert2017great_exp2'],
                [exp3, 'eckert2017great_preferences']]

In [12]:
for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [13]:
# for index in range(1,5):
#     exp = fulldf[fulldf['experiment'] == str(index)]
#     exp = exp.dropna(axis=1, how='all')
#     comp_out_path = os.path.join(out_pathway, 'eckert2017great_exp'+str(index)+'_standardized.csv')
#     exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
#     names = exp.columns.tolist()
#     exp_g = pd.DataFrame(names)
#     exp_g = exp_g.rename(columns={0: "column_name"})
#     exp_g["description"] = ""
#     exp_g=exp_g[["column_name", "description"]]
#     comp_out_path_glossary = os.path.join(out_pathway, 'eckert2017great_exp'+str(index)+'_glossary.csv')
#     exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)